# 02 - CNNs: Cómo las Redes "Ven" Imágenes

Las Redes Convolucionales (CNNs) son la base de la visión por computadora moderna.
En este notebook aprenderemos:

1. Por qué las redes densas no funcionan bien con imágenes
2. La operación de convolución (filtros/kernels)
3. Pooling, stride, padding
4. Arquitecturas clásicas: LeNet → AlexNet → VGG → ResNet → MobileNet
5. Construir y entrenar una CNN en CIFAR-10
6. Visualizar qué aprenden los filtros

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando: {device}")

## 1. El Problema con Redes Densas para Imágenes

Una imagen de 224×224×3 = **150,528** píxeles.
Una capa densa de 1024 neuronas necesitaría 150,528 × 1024 = **154 millones** de parámetros.
¡Solo en la primera capa!

Además, las redes densas no entienden la **estructura espacial** de las imágenes:
- Un gato en la esquina superior izquierda y uno en el centro son completamente diferentes para una red densa
- Pero para nosotros (y para una CNN), es el mismo gato

## 2. La Convolución: Detectar Patrones Locales

Un filtro/kernel es una matriz pequeña (ej. 3×3) que se desliza sobre la imagen.
En cada posición, calcula un producto punto — detectando un patrón específico.

Ventajas:
- **Parameter sharing**: el mismo filtro se aplica en toda la imagen
- **Traslación invariante**: detecta el patrón sin importar dónde esté
- **Pocos parámetros**: un filtro 3×3×3 = solo 27 parámetros

In [ ]:
# Demostración visual de convolución con filtros clásicos
from scipy import signal
from PIL import Image

# Crear una imagen de ejemplo simple
np.random.seed(42)
# Usar un cuadrado blanco sobre fondo negro
img = np.zeros((64, 64))
img[16:48, 16:48] = 1.0  # Cuadrado blanco

# Filtros clásicos
filters = {
    'Bordes\nhorizontales': np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]]),
    'Bordes\nverticales': np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]]),
    'Detector\nde esquinas': np.array([[1, -1, -1], [-1, 2, -1], [-1, -1, 1]]),
    'Gaussian\nblur': (1/16) * np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]),
    'Sharpen': np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]),
}

fig, axes = plt.subplots(2, len(filters) + 1, figsize=(16, 6))

# Fila 1: imagen original + filtros
axes[0, 0].imshow(img, cmap='gray')
axes[0, 0].set_title('Original')

for i, (name, kernel) in enumerate(filters.items()):
    axes[0, i+1].imshow(kernel, cmap='RdBu', vmin=-2, vmax=2)
    axes[0, i+1].set_title(name, fontsize=9)

# Fila 2: resultados de convolución
axes[1, 0].imshow(img, cmap='gray')
axes[1, 0].set_title('Original')

for i, (name, kernel) in enumerate(filters.items()):
    result = signal.convolve2d(img, kernel, mode='same')
    axes[1, i+1].imshow(result, cmap='gray')
    axes[1, i+1].set_title('Resultado')

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('Convolución con diferentes filtros', fontsize=14)
plt.tight_layout()
plt.show()

print("💡 En una CNN, los filtros NO se diseñan a mano.")
print("   La red APRENDE los filtros óptimos durante el entrenamiento.")

## 3. Anatomía de una CNN

```
Imagen → [Conv → ReLU → Pool] × N → Flatten → Dense → Softmax
```

- **Conv2d**: aplica filtros para detectar patrones
- **ReLU**: activación no-lineal
- **MaxPool2d**: reduce tamaño, mantiene features importantes
- **Flatten**: convierte mapa 2D a vector 1D
- **Linear**: clasificación final

Capas tempranas detectan **bordes y texturas**.
Capas profundas detectan **partes de objetos y objetos completos**.

In [ ]:
# Cargar CIFAR-10: 60K imágenes 32x32 en 10 clases
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

classes = ('avión', 'auto', 'pájaro', 'gato', 'ciervo', 'perro', 'rana', 'caballo', 'barco', 'camión')

# Mostrar algunas imágenes
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
raw_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=False)
for i, ax in enumerate(axes.flat):
    img, label = raw_dataset[i]
    ax.imshow(img)
    ax.set_title(classes[label], fontsize=9)
    ax.axis('off')
plt.suptitle('CIFAR-10: 10 clases de imágenes 32x32')
plt.tight_layout()
plt.show()

In [ ]:
class SimpleCNN(nn.Module):
    """CNN simple pero efectiva para CIFAR-10.
    
    Arquitectura:
    Conv(3→32) → ReLU → Conv(32→32) → ReLU → MaxPool
    Conv(32→64) → ReLU → Conv(64→64) → ReLU → MaxPool
    Flatten → Dense(256) → ReLU → Dropout → Dense(10)
    """
    
    def __init__(self, num_classes=10):
        super().__init__()
        
        self.features = nn.Sequential(
            # Bloque 1: 32x32x3 → 16x16x32
            nn.Conv2d(3, 32, 3, padding=1),   # 3x3 filtros, padding para mantener tamaño
            nn.BatchNorm2d(32),                 # Normalización: estabiliza entrenamiento
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                # Reduce 32x32 → 16x16
            nn.Dropout2d(0.25),
            
            # Bloque 2: 16x16x32 → 8x8x64
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                # 16x16 → 8x8
            nn.Dropout2d(0.25),
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN().to(device)

# Contar parámetros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros totales: {total_params:,}")
print(f"Parámetros entrenables: {trainable_params:,}")
print(f"\nArquitectura:")
print(model)

In [ ]:
# Entrenar
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
EPOCHS = 25  # Suficiente para ver el aprendizaje

for epoch in range(EPOCHS):
    # Train
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    train_loss = running_loss / len(trainloader)
    train_acc = correct / total
    
    # Validate
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    val_loss = running_loss / len(testloader)
    val_acc = correct / total
    
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:2d}/{EPOCHS} — "
              f"Train Loss: {train_loss:.3f}, Acc: {train_acc:.3f} | "
              f"Val Loss: {val_loss:.3f}, Acc: {val_acc:.3f}")

In [ ]:
# Gráficas de entrenamiento
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Validation')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Validation')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle(f'CNN en CIFAR-10 — Val Accuracy: {history["val_acc"][-1]:.1%}')
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar los filtros de la primera capa convolucional
# Estos son los patrones que la red aprendió a detectar
first_conv = model.features[0]
filters = first_conv.weight.data.cpu()

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < filters.shape[0]:
        # Normalizar para visualizar
        f = filters[i].permute(1, 2, 0)  # CHW → HWC
        f = (f - f.min()) / (f.max() - f.min())
        ax.imshow(f.numpy())
    ax.axis('off')

plt.suptitle('Filtros aprendidos en la primera capa convolucional\n'
             '(Cada cuadro es un filtro 3x3 que detecta un patrón específico)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar feature maps (activaciones) para una imagen
model.eval()
sample_img, sample_label = testset[0]

# Pasar por las primeras capas
activations = []
x = sample_img.unsqueeze(0).to(device)

for layer in model.features:
    x = layer(x)
    if isinstance(layer, nn.Conv2d):
        activations.append(x.detach().cpu())

# Mostrar activaciones de la primera conv
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
act = activations[0][0]  # Primer batch, primera conv

for i, ax in enumerate(axes.flat):
    if i < act.shape[0]:
        ax.imshow(act[i].numpy(), cmap='viridis')
    ax.axis('off')

plt.suptitle(f'Feature maps de la primera capa conv (clase: {classes[sample_label]})\n'
             f'Cada mapa muestra dónde se activó un filtro específico', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Evolución de Arquitecturas CNN

| Año | Arquitectura | Parámetros | Top-5 Error (ImageNet) | Innovación clave |
|-----|-------------|-----------|------------------------|------------------|
| 1998 | **LeNet-5** | 60K | — | Primera CNN exitosa (dígitos) |
| 2012 | **AlexNet** | 60M | 16.4% | GPU training, ReLU, Dropout |
| 2014 | **VGG-16** | 138M | 7.3% | Filtros 3x3 apilados |
| 2014 | **GoogLeNet** | 6.8M | 6.7% | Inception modules |
| 2015 | **ResNet-50** | 25M | 3.6% | Residual connections (skip) |
| 2017 | **MobileNetV1** | 4.2M | 10.5% | Depthwise separable convolutions |
| 2019 | **EfficientNet** | 5.3M | 2.9% | Scaling compuesto (ancho×profundidad×resolución) |
| 2020 | **MobileNetV3** | 5.4M | 7.5% | NAS + squeeze-excite + h-swish |

### MobileNet: Por qué es ideal para dispositivos móviles

Una convolución normal 3×3 con 32→64 filtros: **18,432** multiplicaciones por posición.

MobileNet usa **Depthwise Separable Convolution** (dos pasos):
1. Depthwise: un filtro 3×3 por canal (32 × 9 = 288)
2. Pointwise: convolución 1×1 para mezclar canales (32 × 64 = 2,048)

Total: 2,336 — ¡**~8x menos** operaciones!

**Siguiente notebook:** Transfer Learning — usar redes pre-entrenadas como base para nuestros modelos.